# WP3 Africa Hazard Ranking — Dataset Notebook 04  
## ThinkHazard! — country-level indicator extraction (hazard levels, 2025)

**Goal (dataset-specific):**  
Extract ThinkHazard! **hazard level** indicators for WP3 Africa countries, harmonise to the shared **master long-format contract**, and upsert into:

`data/intermediate/wp3_country_indicators_master.csv`

---
## What ThinkHazard! provides (important for interpretation)
ThinkHazard! provides **categorical hazard levels** (Very Low / Low / Medium / High), mapped in this dataset to a **1–4 numeric scale**:
- **1 = Very Low**
- **2 = Low**
- **3 = Medium**
- **4 = High**

The hazard level is based on the **expected frequency of exceeding damaging intensity thresholds** over long time horizons (decades–centuries).  
In the WP3 framework, this aligns best with the **Prevalence** dimension (likelihood / frequency).

---
## Project decisions implemented here
- **Use Water scarcity** from ThinkHazard! as a **proxy for Drought** (documented in `notes`).
- For **Wind**, use **both**:
  - Cyclones
  - Tropical Storms  
  (two separate indicators under hazard=Wind; later scoring weights indicators equally)
- Scope = your `ISO3_Regions.csv`.
- Presence is **coverage only**, not scoring.

---
## Inputs (local)
- `data/raw/scope/ISO3_Regions.csv` (or fallback `/mnt/data/ISO3_Regions.csv`)
- ThinkHazard! CSV(s) in `data/raw/ThinkHazard/`  
  Expected file: `WB_THINK_HAZARD_WIDEF.csv` (and optional `WB_THINK_HAZARD_DATADICT.csv`)

**Windows reference path:**  
`C:\pipelines\sewa-multihazar\data\raw\ThinkHazard`  
(In-notebook we use the repo-relative path `data/raw/ThinkHazard/`.)

---
## Outputs
- `data/intermediate/thinkhazard_extracted_2025_long.csv`
- `data/intermediate/thinkhazard_coverage_2025.csv`
- upserted master: `data/intermediate/wp3_country_indicators_long.csv`

---
## Mapping implemented (ThinkHazard! → WP3 hazard taxonomy)
All extracted indicators are mapped to **dimension = Prevalence**.

| ThinkHazard indicator | WP3 hazard | Notes |
|---|---|---|
| Urban Floods | Flash Flooding | proxy: “urban flood” as closest flash/pluvial proxy |
| Floods | Riverine Flooding Continued & Cluster | proxy: “floods” as closest riverine/general flood proxy |
| Coastal Floods | Storm Surge | proxy: coastal flooding as closest available storm surge proxy |
| Cyclones | Wind | wind-storm proxy |
| Tropical Storms | Wind | wind-storm proxy |
| Extreme Heat | Heatwave | direct |
| WildFires | Wildfires | direct |
| Water scarcity | Drought | proxy; explicit note |

Out-of-scope ThinkHazard indicators (Earthquakes / Landslides / Volcanic Activity) are **not appended**.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob

# -----------------------------
# Config — edit paths if needed
# -----------------------------
PROJECT_ROOT = Path(".").resolve()

# Inputs
COUNTRY_SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
THINKHAZARD_DIR    = PROJECT_ROOT.parent / "data" / "raw" / "ThinkHazard"

# Fallbacks for sandbox testing
FALLBACK_SCOPE = Path("/mnt/data/ISO3_Regions.csv")
FALLBACK_WIDEF = Path("/mnt/data/WB_THINK_HAZARD_WIDEF.csv")
FALLBACK_DICT  = Path("/mnt/data/WB_THINK_HAZARD_DATADICT.csv")

if not COUNTRY_SCOPE_PATH.exists() and FALLBACK_SCOPE.exists():
    COUNTRY_SCOPE_PATH = FALLBACK_SCOPE

# Master path (single source of truth)
MASTER_PATH = PROJECT_ROOT.parent / "data" / "intermediate" / "wp3_country_indicators_long.csv"
MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)

# Outputs
OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_LONG     = OUT_DIR / "thinkhazard_extracted_2025_long.csv"
OUT_COVERAGE = OUT_DIR / "thinkhazard_coverage_2025.csv"

print("PROJECT_ROOT:", PROJECT_ROOT.parent)
print("COUNTRY_SCOPE_PATH:", COUNTRY_SCOPE_PATH, "exists:", COUNTRY_SCOPE_PATH.exists())
print("THINKHAZARD_DIR:", THINKHAZARD_DIR, "exists:", THINKHAZARD_DIR.exists())
print("MASTER_PATH:", MASTER_PATH)

PROJECT_ROOT: C:\pipelines\sewa-multihazar
COUNTRY_SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
THINKHAZARD_DIR: C:\pipelines\sewa-multihazar\data\raw\ThinkHazard exists: True
MASTER_PATH: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv


## Global setup — shared long-format contract and upsert

In [3]:
# Canonical enums
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]

SOURCE = "ThinkHazard!"

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]
UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def assert_contract(df: pd.DataFrame) -> None:
    missing_cols = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Contract violation: missing columns {missing_cols}")

    bad_dims = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_dims:
        raise ValueError(f"Unknown dimensions {bad_dims}")

    allowed_haz = set(HAZARDS_9) | {"All Hazards"}
    bad_haz = sorted(set(df["hazard"]) - allowed_haz)
    if bad_haz:
        raise ValueError(f"Unknown hazards {bad_haz}")

    dup = df.duplicated(UPSERT_KEY, keep=False)
    if dup.any():
        raise ValueError("Duplicate UPSERT keys found. Check indicator_id or source rows.")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    assert_contract(new_df)

    if master_path.exists():
        master = pd.read_csv(master_path)
        for col in LONG_COLUMNS:
            if col not in master.columns:
                master[col] = np.nan
        master = master[LONG_COLUMNS]

        master_key = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        new_key    = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~master_key.isin(set(new_key))].copy()

        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()

    out.to_csv(master_path, index=False)
    return out

## Step 1 — Load scope list (ISO3 + region)

In [4]:
scope_raw = pd.read_csv(COUNTRY_SCOPE_PATH)
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None}).ffill()
scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

if scope["iso3"].duplicated().any():
    raise ValueError("Duplicate ISO3 in scope list.")

print("Scope countries:", len(scope))
scope.head()

Scope countries: 47


,iso3,country_name,region
0,NGA,Nigeria,West Africa
1,GHA,Ghana,West Africa
2,SEN,Senegal,West Africa
3,BFA,Burkina Faso,West Africa
4,MLI,Mali,West Africa


## Step 2 — Load ThinkHazard! dataset

This notebook expects `WB_THINK_HAZARD_WIDEF.csv` in `data/raw/ThinkHazard/`.

The dataset is *already long-format*, with a column named `2025` in your current extract.
We detect the latest available year column automatically.

In [5]:
# Locate the main ThinkHazard file
candidates = []
if THINKHAZARD_DIR.exists():
    candidates = sorted(list(THINKHAZARD_DIR.glob("WB_THINK_HAZARD_WIDEF*.csv")))
if not candidates and FALLBACK_WIDEF.exists():
    candidates = [FALLBACK_WIDEF]

if not candidates:
    raise FileNotFoundError("Could not find WB_THINK_HAZARD_WIDEF.csv in repo path or fallback.")

THINKHAZARD_PATH = candidates[0]
print("Using ThinkHazard file:", THINKHAZARD_PATH)

th = pd.read_csv(THINKHAZARD_PATH)

# Standardise columns we need
# REF_AREA = ISO3, REF_AREA_LABEL = name, INDICATOR = code, INDICATOR_LABEL = label
th = th.rename(columns={"REF_AREA":"iso3", "REF_AREA_LABEL":"country_th"})

th["iso3"] = th["iso3"].astype(str).str.strip().str.upper()
th["INDICATOR"] = th["INDICATOR"].astype(str).str.strip()
th["INDICATOR_LABEL"] = th["INDICATOR_LABEL"].astype(str).str.strip()

# Detect year columns
year_cols = [c for c in th.columns if str(c).isdigit()]
if not year_cols:
    raise ValueError("No year columns found in ThinkHazard dataset (expected e.g., '2025').")
YEAR = max(int(c) for c in year_cols)
YEAR_COL = str(YEAR)

print("Detected year columns:", sorted(year_cols))
print("Using YEAR:", YEAR)

# Convert values to numeric
th[YEAR_COL] = pd.to_numeric(th[YEAR_COL], errors="coerce")

th.head()

Using ThinkHazard file: C:\pipelines\sewa-multihazar\data\raw\ThinkHazard\WB_THINK_HAZARD_WIDEF.csv
Detected year columns: ['2025']
Using YEAR: 2025


,FREQ,FREQ_LABEL,iso3,country_th,INDICATOR,INDICATOR_LABEL,UNIT_MEASURE,UNIT_MEASURE_LABEL,DATABASE_ID,DATABASE_ID_LABEL,COMMENT_TS,UNIT_MULT,UNIT_MULT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL,2025
0,A,Annual,AFG,Afghanistan,WB_THINK_HAZARD_CY_LEVEL,Hazard level for Cyclones,1_TO_4,1-4 scale,WB_THINK_HAZARD,ThinkHazard!,Hazard levels for this dataset have been mappe...,0,Units,A,Normal value,PU,Public,2.0
1,A,Annual,AFG,Afghanistan,WB_THINK_HAZARD_DG_LEVEL,Hazard level for Water scarcity,1_TO_4,1-4 scale,WB_THINK_HAZARD,ThinkHazard!,Hazard levels for this dataset have been mappe...,0,Units,A,Normal value,PU,Public,4.0
2,A,Annual,AFG,Afghanistan,WB_THINK_HAZARD_EH_LEVEL,Hazard level for Extreme Heat,1_TO_4,1-4 scale,WB_THINK_HAZARD,ThinkHazard!,Hazard levels for this dataset have been mappe...,0,Units,A,Normal value,PU,Public,4.0
3,A,Annual,AFG,Afghanistan,WB_THINK_HAZARD_EQ_LEVEL,Hazard level for Earthquakes,1_TO_4,1-4 scale,WB_THINK_HAZARD,ThinkHazard!,Hazard levels for this dataset have been mappe...,0,Units,A,Normal value,PU,Public,4.0
4,A,Annual,AFG,Afghanistan,WB_THINK_HAZARD_FL_LEVEL,Hazard level for Floods,1_TO_4,1-4 scale,WB_THINK_HAZARD,ThinkHazard!,Hazard levels for this dataset have been mappe...,0,Units,A,Normal value,PU,Public,4.0


## Step 3 — Dataset diagnostics

In [6]:
ind = th[["INDICATOR","INDICATOR_LABEL"]].drop_duplicates().sort_values("INDICATOR")
print("Unique ThinkHazard indicators:", len(ind))
ind

Unique ThinkHazard indicators: 11


,INDICATOR,INDICATOR_LABEL
9,WB_THINK_HAZARD_CF_LEVEL,Hazard level for Coastal Floods
0,WB_THINK_HAZARD_CY_LEVEL,Hazard level for Cyclones
1,WB_THINK_HAZARD_DG_LEVEL,Hazard level for Water scarcity
2,WB_THINK_HAZARD_EH_LEVEL,Hazard level for Extreme Heat
3,WB_THINK_HAZARD_EQ_LEVEL,Hazard level for Earthquakes
4,WB_THINK_HAZARD_FL_LEVEL,Hazard level for Floods
5,WB_THINK_HAZARD_LS_LEVEL,Hazard level for Landslides
16,WB_THINK_HAZARD_TS_LEVEL,Hazard level for Tropical Storms
6,WB_THINK_HAZARD_UF_LEVEL,Hazard level for Urban Floods
7,WB_THINK_HAZARD_VA_LEVEL,Hazard level for Volcanic Activity


In [7]:
# Coverage vs scope
missing_iso3 = sorted(list(set(scope["iso3"]) - set(th["iso3"])))
print("Scope ISO3 missing in ThinkHazard file:", missing_iso3)

# Which ThinkHazard hazards are out-of-scope for WP3?
out_of_scope = th.loc[th["INDICATOR"].isin([
    "WB_THINK_HAZARD_EQ_LEVEL",
    "WB_THINK_HAZARD_LS_LEVEL",
    "WB_THINK_HAZARD_VA_LEVEL",
]), ["INDICATOR","INDICATOR_LABEL"]].drop_duplicates()

out_of_scope

Scope ISO3 missing in ThinkHazard file: ['CIV', 'COD', 'COG', 'GMB', 'REU', 'STP', 'SWZ']


,INDICATOR,INDICATOR_LABEL
3,WB_THINK_HAZARD_EQ_LEVEL,Hazard level for Earthquakes
5,WB_THINK_HAZARD_LS_LEVEL,Hazard level for Landslides
7,WB_THINK_HAZARD_VA_LEVEL,Hazard level for Volcanic Activity


## Step 4 — Define WP3 mapping (explicit, transparent proxies)

In [8]:
# ThinkHazard → WP3 mapping
# All mapped to dimension=Prevalence (hazard likelihood / long-term exceedance frequency)
MAPPING = [
    # flood split proxies
    dict(indicator="WB_THINK_HAZARD_UF_LEVEL", hazard="Flash Flooding",
         indicator_id="TH.UF_LEVEL", indicator_name="ThinkHazard hazard level: Urban Floods",
         notes="Mapped to Flash Flooding as closest pluvial/urban proxy."),
    dict(indicator="WB_THINK_HAZARD_FL_LEVEL", hazard="Riverine Flooding Continued & Cluster",
         indicator_id="TH.FL_LEVEL", indicator_name="ThinkHazard hazard level: Floods",
         notes="Mapped to Riverine Flooding Continued & Cluster as closest general/riverine proxy."),

    # coastal / storm surge proxy
    dict(indicator="WB_THINK_HAZARD_CF_LEVEL", hazard="Storm Surge",
         indicator_id="TH.CF_LEVEL", indicator_name="ThinkHazard hazard level: Coastal Floods",
         notes="Mapped to Storm Surge as closest available coastal flooding proxy (not a pure surge metric)."),

    # wind: two indicators as agreed
    dict(indicator="WB_THINK_HAZARD_CY_LEVEL", hazard="Wind",
         indicator_id="TH.CY_LEVEL", indicator_name="ThinkHazard hazard level: Cyclones",
         notes="Mapped to Wind; cyclone hazard used as wind-storm proxy."),
    dict(indicator="WB_THINK_HAZARD_TS_LEVEL", hazard="Wind",
         indicator_id="TH.TS_LEVEL", indicator_name="ThinkHazard hazard level: Tropical Storms",
         notes="Mapped to Wind; tropical storms used as wind-storm proxy."),

    # heat, fire
    dict(indicator="WB_THINK_HAZARD_EH_LEVEL", hazard="Heatwave",
         indicator_id="TH.EH_LEVEL", indicator_name="ThinkHazard hazard level: Extreme Heat",
         notes="Mapped to Heatwave."),
    dict(indicator="WB_THINK_HAZARD_WF_LEVEL", hazard="Wildfires",
         indicator_id="TH.WF_LEVEL", indicator_name="ThinkHazard hazard level: WildFires",
         notes="Mapped to Wildfires."),

    # drought proxy
    dict(indicator="WB_THINK_HAZARD_DG_LEVEL", hazard="Drought",
         indicator_id="TH.DG_LEVEL", indicator_name="ThinkHazard hazard level: Water scarcity",
         notes="Mapped to Drought as proxy (Water scarcity is the ThinkHazard drought-related indicator)."),
]

mapping_df = pd.DataFrame(MAPPING)
mapping_df

,indicator,hazard,indicator_id,indicator_name,notes
0,WB_THINK_HAZARD_UF_LEVEL,Flash Flooding,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,Mapped to Flash Flooding as closest pluvial/ur...
1,WB_THINK_HAZARD_FL_LEVEL,Riverine Flooding Continued & Cluster,TH.FL_LEVEL,ThinkHazard hazard level: Floods,Mapped to Riverine Flooding Continued & Cluste...
2,WB_THINK_HAZARD_CF_LEVEL,Storm Surge,TH.CF_LEVEL,ThinkHazard hazard level: Coastal Floods,Mapped to Storm Surge as closest available coa...
3,WB_THINK_HAZARD_CY_LEVEL,Wind,TH.CY_LEVEL,ThinkHazard hazard level: Cyclones,Mapped to Wind; cyclone hazard used as wind-st...
4,WB_THINK_HAZARD_TS_LEVEL,Wind,TH.TS_LEVEL,ThinkHazard hazard level: Tropical Storms,Mapped to Wind; tropical storms used as wind-s...
5,WB_THINK_HAZARD_EH_LEVEL,Heatwave,TH.EH_LEVEL,ThinkHazard hazard level: Extreme Heat,Mapped to Heatwave.
6,WB_THINK_HAZARD_WF_LEVEL,Wildfires,TH.WF_LEVEL,ThinkHazard hazard level: WildFires,Mapped to Wildfires.
7,WB_THINK_HAZARD_DG_LEVEL,Drought,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,Mapped to Drought as proxy (Water scarcity is ...


## Step 5 — Build the long-format extract (appendable to master)

In [9]:
# Filter to only mapped indicators
mapped_codes = set(m["indicator"] for m in MAPPING)
th_sel = th.loc[th["INDICATOR"].isin(mapped_codes)].copy()

# Join to scope to keep only WP3 countries (but retain missing as coverage)
# For extraction, we only output rows where values exist (coverage is separate).
th_scope = scope.merge(th_sel, on="iso3", how="left")

# Build long format
rows = []
for spec in MAPPING:
    col = YEAR_COL
    subset = th_scope.loc[th_scope["INDICATOR"] == spec["indicator"], ["iso3","country_name","region", col]].copy()
    subset = subset.rename(columns={col:"value_raw"})
    subset["hazard"] = spec["hazard"]
    subset["dimension"] = "Prevalence"
    subset["source"] = SOURCE
    subset["indicator_id"] = spec["indicator_id"]
    subset["indicator_name"] = spec["indicator_name"]
    subset["unit_raw"] = "index_1_4"  # 1=Very Low, 2=Low, 3=Medium, 4=High
    subset["year"] = YEAR
    subset["time_window"] = "long_term_screening"
    subset["notes"] = (
        "ThinkHazard hazard levels mapped as very low (1), low (2), medium (3), high (4). "
        + spec["notes"]
    )
    rows.append(subset)

long_df = pd.concat(rows, ignore_index=True)

# Drop missing values from the extract (coverage handled separately)
long_df = long_df.loc[~long_df["value_raw"].isna()].copy()

# Contract + uniqueness checks
long_df = long_df[LONG_COLUMNS]
assert_contract(long_df)

print("Rows extracted:", len(long_df))
long_df.head(12)

Rows extracted: 269


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
0,NGA,Nigeria,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1,GHA,Ghana,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
2,SEN,Senegal,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
3,BFA,Burkina Faso,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
4,MLI,Mali,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
5,NER,Niger,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
6,BEN,Benin,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
7,TGO,Togo,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,3.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
8,SLE,Sierra Leone,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
9,LBR,Liberia,West Africa,Flash Flooding,Prevalence,ThinkHazard!,TH.UF_LEVEL,ThinkHazard hazard level: Urban Floods,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...


## Step 6 — Coverage

In [10]:
cov_rows = []
for spec in MAPPING:
    col = YEAR_COL
    tmp = th_scope.loc[th_scope["INDICATOR"] == spec["indicator"], ["iso3","country_name","region", col]].copy()
    tmp["hazard"] = spec["hazard"]
    tmp["source"] = SOURCE
    tmp["indicator_id"] = spec["indicator_id"]
    tmp["indicator_name"] = spec["indicator_name"]
    tmp["present"] = ~tmp[col].isna()
    tmp["notes"] = "Presence derived from ThinkHazard value-not-missing for year column."
    cov_rows.append(tmp[["iso3","country_name","region","hazard","source","indicator_id","indicator_name","present","notes"]])

coverage = pd.concat(cov_rows, ignore_index=True)
coverage.to_csv(OUT_COVERAGE, index=False)

coverage.groupby(["hazard","present"]).size().unstack(fill_value=0)

present,True
hazard,
Drought,38
Flash Flooding,39
Heatwave,40
Riverine Flooding Continued & Cluster,39
Storm Surge,25
Wildfires,40
Wind,48


## Step 7 — Write extract + upsert into master

In [11]:
long_df.to_csv(OUT_LONG, index=False)

master = upsert_to_master(long_df, MASTER_PATH)

print("Wrote:", OUT_LONG)
print("Wrote:", OUT_COVERAGE)
print("Updated master:", MASTER_PATH, "rows:", len(master))

master.tail(10)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\thinkhazard_extracted_2025_long.csv
Wrote: C:\pipelines\sewa-multihazar\data\intermediate\thinkhazard_coverage_2025.csv
Updated master: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv rows: 1005


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
995,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,3.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
996,ZWE,Zimbabwe,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
997,BWA,Botswana,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
998,NAM,Namibia,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
999,MOZ,Mozambique,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,3.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1000,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,4.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1001,MWI,Malawi,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,2.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1002,LSO,Lesotho,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,2.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1003,MDG,Madagascar,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,1.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...
1004,COM,Comoros,Southern Africaincl. Indian Ocean Islands Country,Drought,Prevalence,ThinkHazard!,TH.DG_LEVEL,ThinkHazard hazard level: Water scarcity,1.0,index_1_4,2025,long_term_screening,ThinkHazard hazard levels mapped as very low (...


## Flags

1) **Taxonomy mismatch is explicitly handled with proxies**
   - Water scarcity → Drought proxy
   - Urban floods → Flash flooding proxy
   - Coastal floods → Storm surge proxy
   - Cyclones + Tropical storms → Wind proxies

2) **This is categorical screening data**
   ThinkHazard hazard levels are categorical and derived from frequency/intensity thresholds.  
   Do not interpret a one-step difference (e.g., 2→3) as a linear change in risk.

3) **Missing coverage is expected**
   Some scope territories may be absent in ThinkHazard country tables; we keep them as missing (coverage flags) with no imputation.